In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# Build clean model (NO Lambda layers)
base_model = MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights=None   # IMPORTANT: no imagenet here
)

base_model.trainable = True

model_clean = models.Sequential([
    layers.Input(shape=(128, 128, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(6, activation='softmax')
])

print("Clean model built.")

Clean model built.


In [5]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# --------------------------------------------------
# 0️⃣ Load the old model safely
# --------------------------------------------------
# Include the custom_objects fix so the Lambda layer doesn't crash the loader
try:
    old_model = tf.keras.models.load_model(
        "mobilenet_model_2_extended.keras",
        custom_objects={'preprocess_input': preprocess_input},
        safe_mode=False
    )
    print("Old model loaded successfully.")
except Exception as e:
    print(f"Failed to load old model: {e}")
    exit()

# (Assuming 'model_clean' is already defined and compiled in your script above this point)

# --------------------------------------------------
# 1️⃣ Transfer MobileNet base weights
# --------------------------------------------------
def get_mobilenet_layer(model):
    """Helper function to safely find the MobileNet base in a model"""
    for layer in model.layers:
        if "mobilenet" in layer.name.lower():
            return layer
    return None

old_base = get_mobilenet_layer(old_model)
new_base = get_mobilenet_layer(model_clean)

if old_base is not None and new_base is not None:
    new_base.set_weights(old_base.get_weights())
    print("✅ MobileNet base weights transferred.")
else:
    print("❌ ERROR: Could not find MobileNet base in one or both models.")

# --------------------------------------------------
# 2️⃣ Transfer classifier head weights (Dense layers)
# --------------------------------------------------
old_dense_layers = [layer for layer in old_model.layers if isinstance(layer, tf.keras.layers.Dense)]
new_dense_layers = [layer for layer in model_clean.layers if isinstance(layer, tf.keras.layers.Dense)]

if len(old_dense_layers) == len(new_dense_layers):
    for i, (new_layer, old_layer) in enumerate(zip(new_dense_layers, old_dense_layers)):
        new_layer.set_weights(old_layer.get_weights())
        print(f"✅ Dense layer {i+1} weights transferred.")
else:
    print(f"❌ Warning: Model mismatch! Old model has {len(old_dense_layers)} Dense layers, "
          f"but clean model has {len(new_dense_layers)}.")

# --------------------------------------------------
# 3️⃣ Save clean model
# --------------------------------------------------
model_clean.save("emotion_model_clean.keras")
print("🚀 Clean model saved successfully as 'emotion_model_clean.keras'.")

Old model loaded successfully.
✅ MobileNet base weights transferred.
✅ Dense layer 1 weights transferred.
✅ Dense layer 2 weights transferred.
🚀 Clean model saved successfully as 'emotion_model_clean.keras'.


In [1]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

model = tf.keras.models.load_model("emotion_model_clean.keras")

emotion_labels = ['angry', 'fear', 'happy', 'neutral', 'sad', 'surprise']

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

cap = cv2.VideoCapture(0)

print("Press q to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]

        face = cv2.resize(face, (128, 128))
        face = cv2.cvtColor(face, cv2.COLOR_GRAY2RGB)
        face = face.astype("float32")
        face = preprocess_input(face)
        face = np.expand_dims(face, axis=0)

        prediction = model.predict(face, verbose=0)
        emotion = emotion_labels[np.argmax(prediction)]

        cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)
        cv2.putText(frame, emotion, (x,y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.9, (0,255,0), 2)

    cv2.imshow("Emotion Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Press q to quit
